# Deep learning with PyTorch

Machine Learning, S8 ELC D1, École Centrale de Lyon, 2027. Yohann De Castro.

**Goal of the session.** You train a dense neural network on images of clothes (Fashion-MNIST) with PyTorch, and
you learn to read its learning curves. You write the training loop yourself. Then you see the effect of the
learning rate, you detect overfitting, you apply early stopping and dropout, and you evaluate the final model once
on the test set.

**Plan (3 hours).**

| Part | Content | Minutes |
| --- | --- | --- |
| 0 | Setup | 5 |
| 1 | Tensors and automatic differentiation | 15 |
| 2 | Data: training, validation and test sets, mini-batches | 15 |
| 3 | A dense network | 15 |
| 4 | Training loop and learning curves | 35 |
| 5 | Learning rate | 20 |
| | Break | 15 |
| 6 | Overfitting and early stopping | 25 |
| 7 | Dropout and final test | 20 |
| 8 | Wrap-up: a checklist for the project | 5 |
| | Slack | 10 |
| 9 | Going further (optional) | after the core parts |

## How to work with this notebook

- Open it in Google Colab. The free CPU runtime is enough: no GPU is needed.
- Run the cells in order, from top to bottom. A cell uses the variables of the cells above it.
- After a restart of the runtime (for example after a disconnection), select the current cell and use
  "Runtime > Run before".
- The code you write replaces two lines `# Answer k.m: hint` and `raise NotImplementedError("Answer k.m")`. Some
  answers sit inside a function: the error then appears only when the function is called.
- The cells that start with `# Check` test your answer. They print `OK` or a message. Do not change them.
- Questions are in grey boxes (**Question k.m.**). Write your answer in the cell **Your answer.** below each one.
- If one epoch of Part 4 takes more than 10 seconds, halve the numbers of epochs in the cell of sizes below.

## PyTorch in ten lines

| Code | What it does |
| --- | --- |
| `torch.tensor(...)` | an array, like `np.array`, that can record operations for the gradient |
| `nn.Linear(d_in, d_out)` | a dense layer $x \mapsto W x + b$ |
| `nn.ReLU()` | the activation $t \mapsto \max(t, 0)$, applied to each coordinate |
| `nn.Sequential(layer1, layer2, ...)` | the composition of the layers |
| `nn.CrossEntropyLoss()` | softmax and cross-entropy, computed from the scores of the network |
| `torch.optim.SGD(model.parameters(), lr=eta)` | stochastic gradient descent on the weights of `model` |
| `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` | reset the gradients, compute them (backpropagation), update the weights |
| `model.train()`, `model.eval()` | training mode, evaluation mode (they differ for dropout) |
| `with torch.no_grad():` | compute without recording the operations (evaluation, manual updates) |
| `loss.item()` | the value of a tensor with one element, as a Python number |

In [ ]:
import copy
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, TensorDataset

print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| numpy", np.__version__)
print("CPU threads used by torch:", torch.get_num_threads())

In [ ]:
SEED = 0


def set_seed(seed):
    # Fix the seeds of Python, NumPy and PyTorch.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

# Sizes used in the notebook. If one epoch takes more than 10 s in Part 4, halve the numbers of epochs.
B = 100                 # mini-batch size (Parts 2 to 4)
B_LR = 32               # mini-batch size of Part 5
N_EPOCHS_MAIN = 10      # Part 4
N_EPOCHS_LR = 4         # Part 5, for each learning rate
N_SUBSET = 5000         # Parts 6 and 7: size of the small training set
B_SUBSET = 50           # Parts 6 and 7: mini-batch size
N_EPOCHS_OVERFIT = 60   # Parts 6 and 7


def moving_average(x, k=50):
    # Means of k consecutive values (the result has len(x) - k + 1 values).
    x = np.asarray(x, dtype=float)
    return np.convolve(x, np.ones(k) / k, mode="valid")

## Part 1. Tensors and automatic differentiation (15 min)

A PyTorch tensor is like a NumPy array. In addition, it can record the operations made on it, so that PyTorch can
compute gradients. The cell below shows the operations used today.

In [ ]:
a = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(a)                  # from NumPy to PyTorch (float64, as a)
u = torch.tensor([[1.0, 0.0], [0.0, 2.0]])  # float32, the type used by the networks
print(t.shape, t.dtype, u.dtype)
print(u @ u)                             # matrix product
print(u + torch.tensor([10.0, 20.0]))    # broadcasting: the vector is added to each row
print((u @ u).numpy())                   # back to NumPy
print(u.sum().item())                    # a tensor with one element, as a Python number

### A network with two weights

Take the network $F(x) = w_2\, r(w_1 x)$, with $r(t) = \max(t, 0)$ (ReLU) and no bias, and the loss
$\ell = \frac12 (F(x) - y)^2$ for one example $(x, y)$. Backpropagation computes $\partial \ell / \partial w_1$ and
$\partial \ell / \partial w_2$ with the chain rule. PyTorch does it for you: create the weights with
`requires_grad=True`, compute the loss, and call `loss.backward()`. The derivative of the loss with respect to `w` is
then in `w.grad`.

**Exercise 1.1.** By hand, compute $\ell$, $\partial \ell / \partial w_1$ and $\partial \ell / \partial w_2$ at
$x = 3$, $y = 1$, $w_1 = 1$, $w_2 = 2$.

**Your answer.**


**Exercise 1.2.** Compute the same loss and the same gradients with autograd.

In [ ]:
x, y = torch.tensor(3.0), torch.tensor(1.0)
# Answer 1.2: create w1 and w2 with requires_grad=True, compute the loss, call backward()
raise NotImplementedError("Answer 1.2")
print("loss", loss.item(), "| dl/dw1", w1.grad.item(), "| dl/dw2", w2.grad.item())

In [ ]:
# Check: the gradients against finite differences
def toy_loss(a, b):
    return 0.5 * (b * max(a * x.item(), 0.0) - y.item()) ** 2


eps = 1e-4
fd1 = (toy_loss(w1.item() + eps, w2.item()) - toy_loss(w1.item() - eps, w2.item())) / (2 * eps)
fd2 = (toy_loss(w1.item(), w2.item() + eps) - toy_loss(w1.item(), w2.item() - eps)) / (2 * eps)
ok = abs(fd1 - w1.grad.item()) < 1e-2 and abs(fd2 - w2.grad.item()) < 1e-2 and abs(loss.item() - toy_loss(1.0, 2.0)) < 1e-6
print("OK" if ok else f"Not OK: finite differences give {fd1:.3f} and {fd2:.3f}")

**Exercise 1.3.** Same computation with $w_1 = -1$, the other values unchanged.

In [ ]:
# Answer 1.3: same as 1.2 with w1 = -1
raise NotImplementedError("Answer 1.3")
print("loss", loss.item(), "| dl/dw1", w1.grad.item(), "| dl/dw2", w2.grad.item())

> **Question 1.1.** With $w_1 = -1$, why are both gradients zero? What happens during training to a ReLU neuron
> whose input is negative for every training example?

**Your answer.**


**Exercise 1.4.** Gradient descent by hand: 20 steps with $\eta = 0.01$, from $w_1 = 1$, $w_2 = 2$. Do the update
inside `with torch.no_grad():`, so that PyTorch does not record it. After the update, reset each gradient with
`w.grad.zero_()`.

In [ ]:
w1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(2.0, requires_grad=True)
eta = 0.01
toy_losses = []
for step in range(20):
    # Answer 1.4: loss and backward(), then, under torch.no_grad(), w -= eta * w.grad and w.grad.zero_() for w1 and w2
    raise NotImplementedError("Answer 1.4")
    toy_losses.append(loss.item())
print("losses:", np.round(toy_losses, 4))
print("final weights: w1 =", round(w1.item(), 3), " w2 =", round(w2.item(), 3))
plt.figure(figsize=(5, 3))
plt.semilogy(toy_losses, "o-")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

> **Question 1.2.** Comment out the line `w.grad.zero_()` and run the cell again. What changes, and why? Then put
> the line back and run the cell again.

**Your answer.**


## Part 2. Data: three sets and mini-batches (15 min)

Fashion-MNIST (Xiao, Rasul and Vollgraf, 2017) has 70,000 grayscale images of 28 x 28 pixels, in 10 classes of
clothes and shoes (7,000 images per class): 60,000 images for training and 10,000 for the test. The next cell
downloads about 30 MB into a folder `data`. It takes a few seconds.

In [ ]:
train_set = torchvision.datasets.FashionMNIST("data", train=True, download=True)
test_set = torchvision.datasets.FashionMNIST("data", train=False, download=True)
images_all, y_all = train_set.data, train_set.targets
images_test, y_test = test_set.data, test_set.targets
# If the download fails, comment out the four lines above and use these ones (Keras is installed on Colab):
# from keras.datasets import fashion_mnist
# (a_train, b_train), (a_test, b_test) = fashion_mnist.load_data()
# images_all, y_all = torch.from_numpy(a_train), torch.from_numpy(b_train).long()
# images_test, y_test = torch.from_numpy(a_test), torch.from_numpy(b_test).long()
print(images_all.shape, images_all.dtype, y_all.shape, y_all.dtype, images_test.shape)

The images are converted once to floats in $[0, 1]$ and flattened to vectors of length $d = 28 \times 28 = 784$.
No transform is applied later, image by image: this keeps the training fast on a CPU.

In [ ]:
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
X_all = images_all.float().div(255).reshape(-1, 784)
X_test_raw = images_test.float().div(255).reshape(-1, 784)
print(X_all.shape, X_all.dtype, "| pixel values in", X_all.min().item(), "to", X_all.max().item())

fig, axes = plt.subplots(2, 10, figsize=(13, 3.2))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(images_all[i], cmap="gray_r")
    ax.set_title(CLASSES[y_all[i]], fontsize=8)
    ax.axis("off")
plt.show()

### Three sets, mini-batches, epochs

- The **training set** is used to fit the weights. The **validation set** is used to choose everything else: the
  learning rate, the number of epochs, the network. The **test set** is used once, at the end, to estimate the
  error of the final model on new data.
- Stochastic gradient descent with mini-batches: at each **iteration**, it takes a mini-batch of $B$ training
  examples, computes the loss on it, and updates the weights once with the gradient of this loss.
- An **epoch** is one pass over the training set, cut into disjoint mini-batches. With $n$ training examples there
  are $\lceil n/B \rceil$ iterations per epoch.

**Exercise 2.1.** Split the 60,000 images at random: 48,000 for training and 12,000 for validation (20 %). Use
`torch.randperm(60000, generator=torch.Generator().manual_seed(SEED))`. The 10,000 test images stay aside until
Part 7.

In [ ]:
# Answer 2.1: random permutation of the 60,000 indices, the first 48,000 for training, the others for validation
raise NotImplementedError("Answer 2.1")
print(X_train_raw.shape, X_val_raw.shape)

In [ ]:
# Check: sizes, no image in both sets, about 4,800 images of each class in the training set
n_common = len(set(idx_train.tolist()) & set(idx_val.tolist()))
print("training images per class:", torch.bincount(y_train, minlength=10).tolist())
print("OK" if len(y_train) == 48000 and len(y_val) == 12000 and n_common == 0 else "Not OK: check the split")

**Exercise 2.2.** Standardize the images. Compute one mean and one standard deviation on the training images (all
pixels together). Then subtract this mean and divide by this standard deviation, for the training, validation and
test images.

In [ ]:
# Answer 2.2: mean and standard deviation of X_train_raw (two numbers), applied to the three sets
raise NotImplementedError("Answer 2.2")
print(f"mean {mean_train:.4f}, standard deviation {std_train:.4f}")

In [ ]:
# Check: mean 0 and standard deviation 1 on the training set; close to these values, but not equal, on validation
print(f"training:   mean {X_train.mean().item():+.4f}, standard deviation {X_train.std().item():.4f}")
print(f"validation: mean {X_val.mean().item():+.4f}, standard deviation {X_val.std().item():.4f}")
ok = abs(X_train.mean().item()) < 1e-4 and abs(X_train.std().item() - 1) < 1e-4 and X_test.shape == (10000, 784)
print("OK" if ok else "Not OK: the statistics must come from the training images only")

A `DataLoader` cuts the training set into mini-batches. With `shuffle=True`, the order changes at each epoch. The
generator seeded with `SEED` makes the order reproducible: two loaders made by `make_loader` give the same
mini-batches in the same order.

In [ ]:
def make_loader(X, y, batch_size):
    # Mini-batches of (X, y), in a random order that is the same for every loader made by this function.
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED))


train_loader = make_loader(X_train, y_train, B)
xb, yb = next(iter(train_loader))
print("one mini-batch:", xb.shape, yb.shape, yb.dtype)

**Exercise 2.3.** Compute the number of iterations per epoch with the formula $\lceil n/B \rceil$.

In [ ]:
# Answer 2.3: n_iter_per_epoch = ceil(n / B), with n the number of training images
raise NotImplementedError("Answer 2.3")
print("iterations per epoch:", n_iter_per_epoch)

In [ ]:
# Check: the formula against the length of the loader
print("OK" if n_iter_per_epoch == len(train_loader) else f"Not OK: the loader has {len(train_loader)} mini-batches")

> **Question 2.1.** Why are the mean and the standard deviation computed on the training images only? What is wrong
> if they are computed on the 70,000 images?

**Your answer.**


> **Question 2.2.** What is the role of each of the three sets? Why must the number of epochs not be chosen on the
> test set?

**Your answer.**


> **Question 2.3.** How many weight updates are made in 10 epochs with $B = 100$? With $B = 32$? Which mini-batch
> size gives the cheapest update, $B = 10$ or $B = 1000$? Which one gives more updates per epoch?

**Your answer.**


## Part 3. A dense network (15 min)

A dense layer computes $h = \phi(W x + b)$, with an activation $\phi$ applied to each coordinate (here ReLU).
`nn.Linear(d_in, d_out)` holds the matrix $W$, of size $d_\text{out} \times d_\text{in}$, and the vector $b$: it has
$d_\text{in} d_\text{out}$ weights and $d_\text{out}$ biases. A network is a composition of such layers.

Output for 10 exclusive classes: the last layer gives 10 scores $z_1, \dots, z_{10}$ (the logits). The softmax turns
them into probabilities $p_k = e^{z_k} / \sum_j e^{z_j}$, which sum to 1. The cross-entropy of an example of class
$y$ is $-\log p_y$. `nn.CrossEntropyLoss` applies the softmax itself: the network outputs the logits and has no
softmax layer.

**Exercise 3.1.** Write the network 784-256-128-10 with ReLU activations: `nn.Linear(784, 256)`, ReLU,
`nn.Linear(256, 128)`, ReLU, `nn.Linear(128, 10)`.

In [ ]:
def make_mlp():
    # The network 784-256-128-10 with ReLU activations; a new random initialization at each call.
    # Answer 3.1: return an nn.Sequential with three nn.Linear layers and two nn.ReLU
    raise NotImplementedError("Answer 3.1")


set_seed(SEED)
model = make_mlp()
print(model)

**Exercise 3.2.** Write a function that counts the parameters of a dense network from the list of its layer sizes.

In [ ]:
def n_params_by_hand(sizes):
    # Number of weights and biases of the dense network with layer sizes `sizes`, for example [784, 256, 128, 10].
    # Answer 3.2: sum over consecutive layers of d_in * d_out + d_out
    raise NotImplementedError("Answer 3.2")


print("by hand:", n_params_by_hand([784, 256, 128, 10]))

In [ ]:
# Check: the count by hand against the count of PyTorch
n_torch = sum(p.numel() for p in model.parameters())
print("PyTorch:", n_torch)
print("OK" if n_params_by_hand([784, 256, 128, 10]) == n_torch else "Not OK")

**Exercise 3.3.** Compute the loss of the untrained network on the mini-batch `xb, yb` of Part 2.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
# Answer 3.3: the logits of the mini-batch, then loss_fn(logits, yb)
raise NotImplementedError("Answer 3.3")
print(f"loss of the untrained network: {loss_init:.3f}   log(10) = {math.log(10):.3f}")

**Exercise 3.4.** For the scores $z = (2, 1, 0.5, -1)$, compute `torch.softmax(z, dim=0)` and `torch.sigmoid(z)`,
and the sum of each.

In [ ]:
z = torch.tensor([2.0, 1.0, 0.5, -1.0])
# Answer 3.4: p_softmax = torch.softmax(z, dim=0), p_sigmoid = torch.sigmoid(z)
raise NotImplementedError("Answer 3.4")
print("softmax:", np.round(p_softmax.numpy(), 3), "sum", round(p_softmax.sum().item(), 3))
print("sigmoid:", np.round(p_sigmoid.numpy(), 3), "sum", round(p_sigmoid.sum().item(), 3))

> **Question 3.1.** Which layer holds most of the parameters? How many parameters has the network 784-512-512-10?

**Your answer.**


> **Question 3.2.** Compare the loss of Exercise 3.3 with $\log 10$. Explain.

**Your answer.**


> **Question 3.3.** In Fashion-MNIST each image has exactly one class. Suppose now that a photo can show several of
> 5 items at once (for example a shirt, a bag and sneakers on the same person). Which output layer and which loss do
> you use? Why not a softmax?

**Your answer.**


> **Question 3.4.** If the two `nn.ReLU()` are removed, which functions of $x$ can the network represent? Does a
> third layer help?

**Your answer.**


## Part 4. Training loop and learning curves (35 min)

One iteration of stochastic gradient descent: compute the scores of the mini-batch (forward pass), the loss, reset
the gradients, compute the gradients (backward pass), update the weights. The function `evaluate` below is given:
it computes the mean loss and the accuracy of a model on a whole set, in evaluation mode and without recording the
operations.

In [ ]:
def evaluate(model, X, y, batch_size=2000):
    # Mean cross-entropy loss and accuracy of the model on (X, y).
    model.eval()
    total_loss, n_correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            logits = model(X[i:i + batch_size])
            total_loss += nn.functional.cross_entropy(logits, y[i:i + batch_size], reduction="sum").item()
            n_correct += (logits.argmax(dim=1) == y[i:i + batch_size]).sum().item()
    return total_loss / len(X), n_correct / len(X)

**Exercise 4.1.** Complete the loop over the mini-batches: forward pass, loss, `optimizer.zero_grad()`,
`loss.backward()`, `optimizer.step()`.

In [ ]:
def train_one_epoch(model, optimizer, loader, scheduler=None):
    # One pass over the loader. Returns the list of the losses of the mini-batches.
    model.train()
    batch_losses = []
    for xb, yb in loader:
        # Answer 4.1: forward pass, loss, zero_grad, backward, step (5 lines)
        raise NotImplementedError("Answer 4.1")
        batch_losses.append(loss.item())
        if scheduler is not None:
            scheduler.step()
        if not math.isfinite(batch_losses[-1]):
            print(f"Non-finite loss at iteration {len(batch_losses)} of this epoch: stop.")
            break
    return batch_losses

At the end of each epoch we measure the loss and the accuracy on the validation set, and on a fixed random subset
of 12,000 training images (`train_eval`). The subset saves time, and 12,000 images give an accurate estimate of the
training loss. The plotting functions are given.

In [ ]:
idx_eval = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(SEED + 1))[:12000]
train_eval = (X_train[idx_eval], y_train[idx_eval])
val = (X_val, y_val)


def plot_batch_losses(histories, iters_per_epoch, k=50, raw=True, logy=False, title="", xmax=None):
    # Loss of each mini-batch (thin line) and its moving average over k iterations (thick line).
    plt.figure(figsize=(10, 4))
    shown = []
    for label, h in histories.items():
        bl = np.array(h["batch_loss"])
        color = None
        if raw:
            color = plt.plot(np.arange(1, len(bl) + 1), bl, lw=0.4, alpha=0.4)[0].get_color()
        if len(bl) >= k:
            ma = moving_average(bl, k)
            plt.plot(np.arange(k, len(bl) + 1), ma, lw=2, color=color, label=f"{label}, moving average over {k} iterations")
            shown.append(ma[:xmax] if xmax is not None else ma)
    n_ep = max(len(h["batch_loss"]) for h in histories.values()) // iters_per_epoch
    for e in range(1, n_ep + 1):
        plt.axvline(e * iters_per_epoch, color="gray", ls=":", lw=0.8)
    if logy:
        plt.yscale("log")
    if xmax is not None:
        plt.xlim(0, xmax)
        values = np.concatenate(shown)
        plt.ylim(0.9 * values.min(), 1.1 * values.max())
    plt.xlabel("iteration (one weight update); dotted lines: end of each epoch")
    plt.ylabel("loss on the mini-batch")
    plt.title(title)
    plt.legend()
    plt.show()


def plot_epoch_curves(histories, best_epochs=None, title=""):
    # Training (dashed) and validation (solid) loss and accuracy at the end of each epoch.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, h in histories.items():
        ep = np.arange(1, len(h["val_loss"]) + 1)
        for ax, key in zip(axes, ("loss", "acc")):
            color = ax.plot(ep, h["train_" + key], "--", label=f"{label}, training")[0].get_color()
            ax.plot(ep, h["val_" + key], "-", color=color, label=f"{label}, validation")
            if best_epochs is not None and label in best_epochs:
                ax.axvline(best_epochs[label], color=color, ls=":", lw=1)
    axes[0].set_ylabel("loss")
    axes[1].set_ylabel("accuracy")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle(title)
    plt.show()

**Exercise 4.2.** Write the function `fit`: for each epoch, train with `train_one_epoch`, then evaluate on
`train_eval` and on `val`, and store everything in the dictionary `history`. If the last loss is not finite, stop.

In [ ]:
def fit(model, optimizer, loader, n_epochs, train_eval, val, scheduler=None):
    history = {"batch_loss": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        # Answer 4.2: train one epoch (extend history["batch_loss"]), stop if not finite, evaluate, store
        raise NotImplementedError("Answer 4.2")
        print(f"epoch {epoch:2d} | training loss {train_loss:.3f} | validation loss {val_loss:.3f} "
              f"| validation accuracy {val_acc:.3f} | {time.time() - t0:.1f} s")
    return history

**Exercise 4.3.** Train a new network `model4` for `N_EPOCHS_MAIN` epochs with `torch.optim.SGD(..., lr=0.05)`. Call
`set_seed(SEED)` first, so that the initialization is the same in every run.

In [ ]:
train_loader = make_loader(X_train, y_train, B)   # a new loader: a second run of this cell gives the same numbers
# Answer 4.3: set_seed, model4 = make_mlp(), an SGD optimizer with lr=0.05, hist4 = fit(...)
raise NotImplementedError("Answer 4.3")

In [ ]:
# Check: one weight update per mini-batch
n_updates = len(hist4["batch_loss"])
print("weight updates:", n_updates)
print("OK" if n_updates == N_EPOCHS_MAIN * len(train_loader) else "Not OK: one update per mini-batch and per epoch")

In [ ]:
plot_batch_losses({"eta = 0.05": hist4}, len(train_loader), title="Loss of each mini-batch, B = 100")
bl = np.array(hist4["batch_loss"])
print(f"first mini-batch loss {bl[0]:.3f}; first iteration below 0.6: {np.argmax(bl < 0.6) + 1}; "
      f"last epoch: between {bl[-len(train_loader):].min():.2f} and {bl[-len(train_loader):].max():.2f}")

In [ ]:
plot_epoch_curves({"eta = 0.05": hist4}, title="Curves per epoch, 48,000 training images")

> **Question 4.1.** The loss of the mini-batches goes up and down from one iteration to the next. Is the gradient
> computation wrong? Is the network overfitting? Explain.

**Your answer.**


> **Question 4.2.** Describe the curves per epoch. Is there overfitting after 10 epochs? Why is the validation loss
> slightly above the training loss? (You should find a validation accuracy between 87 and 90 % after 10 epochs.)

**Your answer.**


**Exercise 4.4.** Noise of the mini-batch loss. The trained network `model4` is frozen. For $B$ in 1, 10, 100 and
1000, draw 200 random mini-batches of size $B$ from the training set, compute the loss of each, and the standard
deviation of these 200 values.

In [ ]:
gen = torch.Generator().manual_seed(SEED)
noise_std = {}
model4.eval()
with torch.no_grad():
    # Answer 4.4: for each b in (1, 10, 100, 1000): 200 losses on random indices torch.randint(0, n, (b,), generator=gen), then np.std
    raise NotImplementedError("Answer 4.4")
sizes = np.array(list(noise_std))
stds = np.array(list(noise_std.values()))
slope = np.polyfit(np.log(sizes), np.log(stds), 1)[0]
print("standard deviations:", {b: round(float(s), 4) for b, s in noise_std.items()})
print(f"fitted slope in log-log scale: {slope:.2f}")
plt.figure(figsize=(5, 3.5))
plt.loglog(sizes, stds, "o-", label="standard deviation of the mini-batch loss")
plt.loglog(sizes, stds[0] * sizes ** -0.5, "k:", label="slope -1/2")
plt.xlabel("mini-batch size B")
plt.legend()
plt.show()

> **Question 4.3.** Explain the slope $-1/2$. Would mini-batches of one image remove the oscillations? Why do we
> still use small mini-batches? (You should find a fitted slope between -0.6 and -0.45.)

**Your answer.**


## Part 5. Learning rate (20 min)

The learning rate $\eta$ is the size of the step: $w \leftarrow w - \eta\, g$, with $g$ the gradient of the loss of
the mini-batch. Too small, the loss decreases slowly. Too large, the iterates jump from one side of the valley to
the other, and the noise of the mini-batch gradient is multiplied by $\eta$: the loss stops decreasing, or explodes.
A remedy is a step that decreases over the iterations, $\eta_k = \eta_0 / (1 + k / k_0)$, where $k$ is the number of
the iteration (here $k_0$ is the number of iterations of one epoch: the step is halved after one epoch).

In this part the mini-batches have $B = 32$ images (`B_LR`), so 1,500 iterations per epoch. Each run starts from
the same initialization and sees the mini-batches in the same order: only $\eta$ differs.

**Exercise 5.1.** Write `run_lr(eta)`: `set_seed(SEED)`, a new network, a new SGD optimizer with step `eta`, a new
loader with mini-batches of `B_LR`, and `fit` for `n_epochs` epochs.

In [ ]:
def run_lr(eta, n_epochs=N_EPOCHS_LR):
    # Answer 5.1: set_seed(SEED), model, optimizer with lr=eta, loader = make_loader(X_train, y_train, B_LR), return fit(...)
    raise NotImplementedError("Answer 5.1")


hist_lr = {}

In [ ]:
hist_lr[0.001] = run_lr(0.001)

In [ ]:
hist_lr[0.05] = run_lr(0.05)

In [ ]:
hist_lr[0.5] = run_lr(0.5)

In [ ]:
iters_lr = math.ceil(len(X_train) / B_LR)


def lr_summary(histories):
    for label, h in histories.items():
        bl = np.array(h["batch_loss"])
        print(f"{label:>22} | mean of the first 20 iterations {bl[:20].mean():.3f} | mean of the last 50 "
              f"iterations {bl[-50:].mean():.3f} | final validation loss {h['val_loss'][-1]:.3f}")


lr_summary({f"eta = {eta}": h for eta, h in hist_lr.items()})
plot_batch_losses({f"eta = {eta}": h for eta, h in hist_lr.items()}, iters_lr, raw=False, logy=True,
                  title="Moving average of the mini-batch loss, B = 32")
plot_batch_losses({f"eta = {eta}": h for eta, h in hist_lr.items()}, iters_lr, k=5, raw=False, logy=True,
                  title="Zoom on the first 100 iterations (moving average over 5 iterations)", xmax=100)

In [ ]:
plot_epoch_curves({f"eta = {eta}": h for eta, h in hist_lr.items()}, title="Curves per epoch, B = 32")

> **Question 5.1.** Match each curve with "too small", "good" and "too large". Explain the plateau of the largest
> step. (You should find, for the mean of the last 50 iterations: about 0.6 for $\eta = 0.001$, between 0.25 and 0.35 for $\eta = 0.05$, between 0.35 and 0.5 for $\eta = 0.5$.)

**Your answer.**


**Exercise 5.2.** Train again with $\eta_0 = 0.5$, with the decreasing step
`torch.optim.lr_scheduler.LambdaLR(optimizer, lambda k: 1 / (1 + k / len(loader)))`, passed to `fit`.
`train_one_epoch` already calls `scheduler.step()` after each update.

In [ ]:
# Answer 5.2: as in run_lr with eta = 0.5, plus a LambdaLR scheduler passed to fit
raise NotImplementedError("Answer 5.2")

In [ ]:
compare = {"eta = 0.05": hist_lr[0.05], "eta = 0.5, constant": hist_lr[0.5], "eta = 0.5, decreasing": hist_lr_decreasing}
lr_summary(compare)
plot_batch_losses(compare, iters_lr, raw=False, logy=True, title="Constant and decreasing step, B = 32")

> **Question 5.2.** Describe in one sentence what the training loss looks like when the step is too large, and what
> you do then.

**Your answer.**


**Exercise 5.3.** Train with $\eta = 5$ for one epoch.

In [ ]:
# Answer 5.3: run_lr with eta = 5 and one epoch
raise NotImplementedError("Answer 5.3")
print("mini-batch losses:", np.round(hist_lr5["batch_loss"][:12], 2))

> **Question 5.3.** What happens with $\eta = 5$? What would a loss that stays at about 2.30 mean (compare with
> Question 3.2)?

**Your answer.**


## Part 6. Overfitting and early stopping (25 min)

A network overfits when it fits the noise of the training set: the training loss keeps decreasing while the
validation loss goes up. **Early stopping** keeps the weights of the epoch with the lowest validation loss. With a
patience $P$, training stops when the validation loss has not improved for $P$ epochs.

To see overfitting within a few minutes, we train on a small training set: the first $5{,}000$ training images,
with mini-batches of 50 (100 iterations per epoch), for 60 epochs. The validation set is unchanged. We use SGD with
momentum $\gamma = 0.9$: the update uses a moving average of the past gradients, which speeds up training. PyTorch
writes it $v \leftarrow \gamma v + g$, $w \leftarrow w - \eta v$. The lecture writes
$z \leftarrow \gamma z + (1 - \gamma) g$, $w \leftarrow w - \eta z$. So a PyTorch step $\eta$ with $\gamma = 0.9$
acts like a step $10\, \eta$ in the convention of the lecture.

In [ ]:
X_small, y_small = X_train[:N_SUBSET], y_train[:N_SUBSET]
small_eval = (X_small, y_small)        # the training loss is measured on the 5,000 images
print("iterations per epoch:", len(make_loader(X_small, y_small, B_SUBSET)))

**Exercise 6.1.** Complete `fit_early_stopping`. After the evaluation of each epoch: if the validation loss is the
best so far, keep it, keep the epoch and a copy of the weights `copy.deepcopy(model.state_dict())`, and reset the
counter of epochs without improvement; otherwise increase the counter. If `patience` is not `None` and the counter
reaches `patience`, stop. At the end, the function loads the best weights.

In [ ]:
def fit_early_stopping(model, optimizer, loader, n_epochs, train_eval, val, patience=None, print_every=5):
    history = {"batch_loss": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_loss, best_epoch, best_state, n_bad = math.inf, 0, None, 0
    for epoch in range(1, n_epochs + 1):
        history["batch_loss"] += train_one_epoch(model, optimizer, loader)
        for name, (X, y) in (("train", train_eval), ("val", val)):
            loss, acc = evaluate(model, X, y)
            history[name + "_loss"].append(loss)
            history[name + "_acc"].append(acc)
        if epoch % print_every == 0:
            print(f"epoch {epoch:2d} | training loss {history['train_loss'][-1]:.3f} "
                  f"| validation loss {history['val_loss'][-1]:.3f} | validation accuracy {history['val_acc'][-1]:.3f}")
        # Answer 6.1: keep the best loss, epoch and weights; count the epochs without improvement; stop after patience
        raise NotImplementedError("Answer 6.1")
    model.load_state_dict(best_state)
    print(f"best epoch {best_epoch}, validation loss {best_loss:.3f}; its weights are loaded")
    return history, best_epoch

**Exercise 6.2.** Train a new network `model6` on the small training set for `N_EPOCHS_OVERFIT` epochs, with
`patience=None` to see the whole curves. Optimizer: `torch.optim.SGD(model6.parameters(), lr=0.01, momentum=0.9)`.
Loader: `make_loader(X_small, y_small, B_SUBSET)`. Training loss measured on `small_eval`.

In [ ]:
# Answer 6.2: set_seed, model6, SGD with momentum, hist6, best6 = fit_early_stopping(..., patience=None)
raise NotImplementedError("Answer 6.2")

In [ ]:
plot_epoch_curves({"no dropout": hist6}, best_epochs={"no dropout": best6},
                  title="5,000 training images, momentum 0.9 (dotted line: best epoch)")

> **Question 6.1.** Describe the two curves. From which epoch does the network overfit? Is it underfitting? Would a
> larger $\eta$ lower the validation loss? (You should find the lowest validation loss, about 0.42, between epochs
> 5 and 15, and a validation loss of about 0.9 at epoch 60.)

**Your answer.**


> **Question 6.2.** The validation loss rises a lot while the validation accuracy hardly changes. Explain.

**Your answer.**


**Exercise 6.3.** From `hist6["val_loss"]`, find the epoch where training would stop with a patience of 5: the first
epoch that comes 5 epochs after the last improvement.

In [ ]:
vl6 = hist6["val_loss"]
# Answer 6.3: best epoch so far after each epoch, then the first epoch e with e - best >= 5
raise NotImplementedError("Answer 6.3")
print("with patience 5, training stops at epoch", stop5)

In [ ]:
# Check: the restored model has the lowest validation loss of the curve; a run with patience 5 stops at the same epoch
restored_loss = evaluate(model6, *val)[0]
set_seed(SEED)
m = make_mlp()
h, _ = fit_early_stopping(m, torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9),
                          make_loader(X_small, y_small, B_SUBSET), N_EPOCHS_OVERFIT, small_eval, val, patience=5,
                          print_every=100)
ok = abs(restored_loss - min(vl6)) < 1e-6 and len(h["val_loss"]) == stop5
print("OK" if ok else f"Not OK: restored loss {restored_loss:.4f}, minimum {min(vl6):.4f}, stop {len(h['val_loss'])}")

> **Question 6.3.** Compare with Part 4 (48,000 images, no overfitting in 10 epochs). What is the effect of the size
> of the training set?

**Your answer.**


> **Question 6.4.** Why do we use one validation set here, and not 5-fold cross-validation? When is cross-validation
> the better choice?

**Your answer.**


## Part 7. Dropout and final test (20 min)

Dropout with rate $p$: in training mode, each hidden unit is set to 0 with probability $p$, independently at each
iteration, and the other units are multiplied by $1/(1-p)$. In evaluation mode, dropout does nothing. The network
cannot rely on a few units: this reduces overfitting. In PyTorch, `nn.Dropout(p)` is a layer placed after an
activation.

**Exercise 7.1.** Write the network of Part 3 with `nn.Dropout(p)` after each hidden ReLU.

In [ ]:
def make_mlp_dropout(p=0.5):
    # Answer 7.1: the network of make_mlp with nn.Dropout(p) after each of the two ReLU
    raise NotImplementedError("Answer 7.1")


print(make_mlp_dropout())

In [ ]:
# Check: dropout adds no parameter
n_drop = sum(p.numel() for p in make_mlp_dropout().parameters())
n_plain = sum(p.numel() for p in make_mlp().parameters())
print("parameters with dropout:", n_drop, "| without:", n_plain)
print("OK" if n_drop == n_plain and any(isinstance(l, nn.Dropout) for l in make_mlp_dropout()) else "Not OK")

**Exercise 7.2.** Pass one validation image twice through a network with dropout in `model.train()` mode, then twice
in `model.eval()` mode. Compare the outputs with `torch.equal`.

In [ ]:
set_seed(SEED)
model_d = make_mlp_dropout()
x1 = X_val[:1]
# Answer 7.2: two outputs in train mode (train_a, train_b), two in eval mode (eval_a, eval_b)
raise NotImplementedError("Answer 7.2")
print("training mode, same output twice:  ", torch.equal(train_a, train_b))
print("evaluation mode, same output twice:", torch.equal(eval_a, eval_b))

**Exercise 7.3.** Train `model7 = make_mlp_dropout(0.5)` in the setting of Part 6 (same seed, optimizer, loader,
epochs, `patience=None`).

In [ ]:
# Answer 7.3: as in 6.2, with make_mlp_dropout(0.5); results in hist7, best7
raise NotImplementedError("Answer 7.3")

In [ ]:
plot_epoch_curves({"no dropout": hist6, "dropout 0.5": hist7}, best_epochs={"no dropout": best6, "dropout 0.5": best7},
                  title="5,000 training images, with and without dropout (dotted lines: best epochs)")
for label, h in (("no dropout", hist6), ("dropout 0.5", hist7)):
    v = np.array(h["val_loss"])
    print(f"{label:>12} | lowest validation loss {v.min():.3f} at epoch {v.argmin() + 1} | at epoch {len(v)}: "
          f"{v[-1]:.3f} (rise {v[-1] - v.min():.3f}) | validation accuracy at the end {h['val_acc'][-1]:.3f}")

> **Question 7.1.** What does dropout change in the curves? Why is it called a regularization? Does it add
> parameters? Is it applied when the network predicts on new data? (You should find a rise of the validation loss
> after its minimum at least three times smaller with dropout.)

**Your answer.**


> **Question 7.2.** Why must `evaluate` call `model.eval()`, and `train_one_epoch` call `model.train()`?

**Your answer.**


**Exercise 7.4.** Among `model4` (Part 4), `model6` and `model7` (Parts 6 and 7, restored to their best epoch),
choose the model with the lowest validation loss. Evaluate it once on the test set.

In [ ]:
candidates = {"Part 4": model4, "Part 6": model6, "Part 7": model7}
for name, m in candidates.items():
    print(f"{name}: validation loss {evaluate(m, *val)[0]:.3f}")
# Answer 7.4: final_name = the name with the lowest validation loss, final_model, then test_loss, test_acc
raise NotImplementedError("Answer 7.4")
print(f"chosen: {final_name} | test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")

> **Question 7.3.** Why is the test set used only once? Is the validation loss of the chosen model an unbiased
> estimate of its loss on new data? (You should find a test accuracy between 86 and 89 %.)

**Your answer.**


## Part 8. Wrap-up: a checklist for the project (5 min)

1. Split the data into training, validation and test sets, with a fixed seed.
2. Fit the preprocessing (here the standardization) on the training set only.
3. Tune the learning rate first, on the curves of the first epochs.
4. Plot the training and validation losses per epoch before any other tuning.
5. Use early stopping and keep the best weights.
6. If the gap between the two curves grows, regularize (dropout, weight decay) or get more data.
7. Evaluate the final model once on the test set.

> **Question 8.1.** A training set has $n = 20{,}000$ examples. How many weight updates are made in 8 epochs with
> mini-batches of $B = 64$?

**Your answer.**


> **Question 8.2.** The validation loss rises after epoch 20 while the training loss keeps decreasing. What do you do?

**Your answer.**


> **Question 8.3.** The loss of the mini-batches is jagged from one iteration to the next. Is it a bug?

**Your answer.**


## Part 9. Going further (optional)

The subsections are independent of each other. They use the functions and the data of Parts 2 to 7. In order of
priority: 9.1, 9.2, 9.3, then the others.

### 9.1 Vanishing gradient

We compare two deep networks, with 15 hidden layers of 64 units each. They differ only by the activation: ReLU in
one, the sigmoid $\sigma(t) = 1/(1 + e^{-t})$ in the other. Both are initialized in the same way (He
initialization, zero biases).

In [ ]:
def make_deep_mlp(activation, depth=15, width=64):
    sizes = [784] + [width] * depth + [10]
    layers = []
    for i in range(len(sizes) - 1):
        linear = nn.Linear(sizes[i], sizes[i + 1])
        nn.init.kaiming_normal_(linear.weight)
        nn.init.zeros_(linear.bias)
        layers.append(linear)
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)


print("parameters:", sum(p.numel() for p in make_deep_mlp(nn.ReLU).parameters()))

**Exercise 9.1a.** Write `weight_grad_norms`: one forward pass and one backward pass on a mini-batch, then the norm
of the gradient of the weight matrix of each linear layer, from the input to the output.

In [ ]:
def weight_grad_norms(model, xb, yb):
    # Answer 9.1a: model.zero_grad(), loss_fn(model(xb), yb).backward(), then layer.weight.grad.norm() for each nn.Linear
    raise NotImplementedError("Answer 9.1a")


xb, yb = next(iter(make_loader(X_train, y_train, B)))
deep, norms = {}, {}
for name, activation in (("ReLU", nn.ReLU), ("sigmoid", nn.Sigmoid)):
    set_seed(SEED)
    deep[name] = make_deep_mlp(activation)
    norms[name] = weight_grad_norms(deep[name], xb, yb)
    print(f"{name:>8}: first layer {norms[name][0]:.1e}, last layer {norms[name][-1]:.1e}, "
          f"ratio first / last {norms[name][0] / norms[name][-1]:.1e}")
plt.figure(figsize=(6, 3.5))
for name in norms:
    plt.semilogy(np.arange(1, len(norms[name]) + 1), norms[name], "o-", label=name)
plt.xlabel("layer (1 = next to the input)")
plt.ylabel("norm of the gradient of the weights")
plt.legend()
plt.show()

**Exercise 9.1b.** Train both networks for 2 epochs with `torch.optim.SGD(..., lr=0.05)`. For each linear layer,
compute the relative change of its weights $\|W_l^\text{end} - W_l^\text{start}\| / \|W_l^\text{start}\|$.

In [ ]:
rel_change, hist_deep = {}, {}
for name in deep:
    linears = [layer for layer in deep[name] if isinstance(layer, nn.Linear)]
    # Answer 9.1b: copy the initial weights, fit 2 epochs (hist_deep[name]), then the relative change of each layer
    raise NotImplementedError("Answer 9.1b")
plt.figure(figsize=(6, 3.5))
for name in rel_change:
    plt.semilogy(np.arange(1, len(rel_change[name]) + 1), rel_change[name], "o-", label=name)
plt.xlabel("layer (1 = next to the input)")
plt.ylabel("relative change of the weights")
plt.legend()
plt.show()

> **Question 9.1.** (a) Use $\sigma'(t) = \sigma(t)(1 - \sigma(t)) \le 1/4$ and the chain rule to explain the first
> figure. (b) Why does ReLU help? (c) Would a smaller $\eta$ help the first layers of the sigmoid network? Would
> dropout help?

**Your answer.**


### 9.2 Identity activations

**Exercise 9.2.** Train the network 784-256-128-10 without its two ReLU (three linear layers), as `model4` (same
seed, SGD with `lr=0.05`, mini-batches of `B`), for 4 epochs.

In [ ]:
# Answer 9.2: set_seed, linear3 = nn.Sequential of the three nn.Linear without activation, hist_linear3 = fit(... 4 epochs)
raise NotImplementedError("Answer 9.2")

In [ ]:
# Check: the network is one affine map x -> W x + b, with W = W3 W2 W1 and b = W3 (W2 b1 + b2) + b3
l1, l2, l3 = linear3
with torch.no_grad():
    W = l3.weight @ l2.weight @ l1.weight
    b = l3.weight @ (l2.weight @ l1.bias + l2.bias) + l3.bias
    print("OK" if torch.allclose(linear3(X_val[:5]), X_val[:5] @ W.T + b, atol=1e-4) else "Not OK")

In [ ]:
set_seed(SEED)
linear6 = nn.Sequential(nn.Linear(784, 256), nn.Linear(256, 128), nn.Linear(128, 128), nn.Linear(128, 128),
                        nn.Linear(128, 128), nn.Linear(128, 10))
hist_linear6 = fit(linear6, torch.optim.SGD(linear6.parameters(), lr=0.05), make_loader(X_train, y_train, B), 4,
                   train_eval, val)
print(f"validation accuracy after 4 epochs: 3 linear layers {hist_linear3['val_acc'][-1]:.3f} | "
      f"6 linear layers {hist_linear6['val_acc'][-1]:.3f} | network of Part 4 (ReLU) {hist4['val_acc'][3]:.3f}")

> **Question 9.2.** Compare the three validation accuracies. Does a deeper network without activation do better?

**Your answer.**


### 9.3 Several labels per image

The task below is artificial: we derive from the classes three attributes that are not exclusive. "Upper body"
(T-shirt/top, Pullover, Coat, Shirt), "warm" (Pullover, Coat, Ankle boot) and "footwear" (Sandal, Sneaker, Ankle
boot). An ankle boot is warm and footwear; a trouser has none of the three attributes. The network has 3 outputs,
one sigmoid each, and the loss is the binary cross-entropy of each output (`nn.BCEWithLogitsLoss`, which applies the
sigmoids itself).

In [ ]:
ATTRIBUTES = ["upper body", "warm", "footwear"]
ATTR = torch.zeros(10, 3)
ATTR[[0, 2, 4, 6], 0] = 1   # upper body
ATTR[[2, 4, 9], 1] = 1      # warm
ATTR[[5, 7, 9], 2] = 1      # footwear
Y_train_attr, Y_val_attr = ATTR[y_train], ATTR[y_val]
print("attributes of an ankle boot:", ATTR[9].tolist())

**Exercise 9.3.** Train a network 784-256-128-3 with ReLU for 2 epochs on `(X_train, Y_train_attr)`, with
`nn.BCEWithLogitsLoss()` and SGD (`lr=0.05`). Write the training loop directly (the function `fit` uses the
cross-entropy of Part 3).

In [ ]:
set_seed(SEED)
model_ml = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 3))
bce = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_ml.parameters(), lr=0.05)
loader_ml = make_loader(X_train, Y_train_attr, B)
# Answer 9.3: 2 epochs; for each mini-batch: loss = bce(model_ml(xb), yb), zero_grad, backward, step
raise NotImplementedError("Answer 9.3")
model_ml.eval()
with torch.no_grad():
    proba = torch.sigmoid(model_ml(X_val))
acc_attr = ((proba > 0.5).float() == Y_val_attr).float().mean(dim=0)
for name, a in zip(ATTRIBUTES, acc_attr):
    print(f"validation accuracy for '{name}': {a.item():.3f}")
i_boot = int((y_val == 9).nonzero()[0])
print("probabilities for a validation ankle boot:", np.round(proba[i_boot].numpy(), 3),
      "| sum", round(proba[i_boot].sum().item(), 3))

> **Question 9.3.** Why do the three probabilities of the ankle boot not sum to 1? What would go wrong with a softmax
> over the three outputs?

**Your answer.**


### 9.4 Weight decay

Weight decay adds $\frac{\lambda}{2} \|w\|^2$ to the loss: each SGD step also pulls the weights towards 0. It is the
penalty of ridge regression. In PyTorch it is the argument `weight_decay=lam` of the optimizer.

**Exercise 9.4.** Train `make_mlp()` in the setting of Part 6 with `weight_decay=1e-3`.

In [ ]:
# Answer 9.4: as in 6.2 with weight_decay=1e-3 in the optimizer; results in hist_wd, best_wd
raise NotImplementedError("Answer 9.4")

In [ ]:
plot_epoch_curves({"no regularization": hist6, "dropout 0.5": hist7, "weight decay 1e-3": hist_wd},
                  title="5,000 training images: two regularizations")
for label, h in (("no regularization", hist6), ("dropout 0.5", hist7), ("weight decay 1e-3", hist_wd)):
    v = np.array(h["val_loss"])
    print(f"{label:>18} | lowest validation loss {v.min():.3f} at epoch {v.argmin() + 1} | at epoch {len(v)}: {v[-1]:.3f}")

> **Question 9.4.** Compare weight decay with dropout on these curves. What is the link with ridge regression?

**Your answer.**


### 9.5 Confusion matrix

The confusion matrix of the final model on the test set describes its errors. Nothing is chosen with it, so the test
set keeps its role.

**Exercise 9.5.** Compute the predictions of `final_model` on `X_test` and the confusion matrix `cm`, of size
10 x 10, where `cm[i, j]` counts the test images of class `i` predicted in class `j`.

In [ ]:
final_model.eval()
with torch.no_grad():
    # Answer 9.5: pred = argmax of the scores; cm = torch.bincount(10 * y_test + pred, minlength=100).reshape(10, 10)
    raise NotImplementedError("Answer 9.5")
plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=60)
plt.yticks(range(10), CLASSES)
for i in range(10):
    for j in range(10):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 500 else "black")
plt.xlabel("predicted class")
plt.ylabel("true class")
plt.show()
off = cm.clone().fill_diagonal_(0)
for k in torch.argsort(off.flatten(), descending=True)[:3].tolist():
    print(f"{CLASSES[k // 10]} predicted as {CLASSES[k % 10]}: {int(off.flatten()[k])} images")

> **Question 9.5.** Which classes are confused? Is it surprising?

**Your answer.**


### 9.6 Mini-batch size

**Exercise 9.6.** Train `make_mlp()` for one epoch with SGD (`lr=0.05`) and mini-batches of $B = 10$, then of
$B = 1000$. Store in `results_b[b]` the time, the number of updates and the validation loss after one epoch.

In [ ]:
results_b = {}
for b in (10, 1000):
    # Answer 9.6: set_seed, a new model and optimizer, fit for 1 epoch with make_loader(X_train, y_train, b), timed
    raise NotImplementedError("Answer 9.6")
for b, (t, n, v) in results_b.items():
    print(f"B = {b:4d}: {n} updates in {t:.1f} s, validation loss after one epoch {v:.3f}")

> **Question 9.6.** Compare the time per epoch and the loss after one epoch. Which size would you choose?

**Your answer.**
